## Imports & setting up:

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM
from typing import Optional, Iterable, Dict, List
import os

import pandas as pd
import numpy as np
import warnings
import re
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch.nn as nn
from skorch.helper import SliceDataset
from datetime import datetime
from skorch.callbacks import EarlyStopping, LRScheduler, Checkpoint
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset
import pickle 
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl
from matplotlib.ticker import FuncFormatter

from matplotlib.lines import Line2D
import scipy.stats as scipy_stats
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"CROSS_REGION"
print(f"Base directory for data: {BASE_DIR}")

MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# --- plot KDEs ---
COL_LABELS = {
    "t2m": "2m temperature",
    "tp": "Precipitation",
    "slhf": "Latent heat flux",
    "sshf": "Sensible heat flux",
    "ssrd": "Solar radiation",
    "fal": "Albedo",
    "str": "Thermal radiation",
    "ELEVATION_DIFFERENCE": "Elevation diff. (m)",
    "aspect": "Aspect (°)",
    "slope": "Slope (°)",
    "svf": "Sky-view factor",
}

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc"),
    'csv_path':
    os.path.join(cfg.dataPath, 'WGMS')
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")


In [ ]:
# -----------------------------------------------------------------------
# Nature-compliant colorblind-safe palette
# Based on: https://research-figure-guide.nature.com/figures/building-and-exporting-figure-panels/
# -----------------------------------------------------------------------

NATURE_PALETTE = {
    "black": "#000000",
    "orange": "#e69f00",
    "sky_blue": "#56b4e9",
    "bluish_green": "#009e73",
    "yellow": "#f0e442",
    "blue": "#0072b2",
    "vermillion": "#d55e00",
    "reddish_purple": "#cc79a7",
}

# Convenient ordered list for sequential assignment
NATURE_COLORS = list(NATURE_PALETTE.values())

# Nature figure specs (for reference when setting figsize)
NATURE_SPECS = {
    "single_col_mm": 89,
    "double_col_mm": 183,
    "max_height_mm": 170,
    "font_min_pt": 5,
    "font_max_pt": 7,
    "font_family": "Arial",
    "dpi": 300,
}


def nature_figsize(cols=1, height_mm=80):
    """Return figsize in inches for Nature single or double column."""
    width_mm = NATURE_SPECS["single_col_mm"] if cols == 1 \
               else NATURE_SPECS["double_col_mm"]
    return (width_mm / 25.4, height_mm / 25.4)


def apply_nature_style(ax, fontsize=6, box=False):
    if box:
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.4)
    else:
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["bottom", "left"]].set_linewidth(0.5)
    ax.tick_params(labelsize=fontsize, width=0.5, length=3)
    ax.xaxis.label.set_size(fontsize)
    ax.yaxis.label.set_size(fontsize)
    ax.title.set_size(fontsize + 1)
    ax.grid(color="#e0e0e0", linewidth=0.3, zorder=0)
    ax.set_axisbelow(True)


mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 6,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "xtick.major.size": 3,
    "ytick.major.width": 0.5,
    "ytick.major.size": 3,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

nature_seq = LinearSegmentedColormap.from_list(
    "nature_seq",
    [
        NATURE_PALETTE["sky_blue"], NATURE_PALETTE["blue"],
        NATURE_PALETTE["black"]
    ],
)


## Load glacier grids:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head(2)

### Switzerland:

In [ ]:
df_CH = load_stakes(cfg, "CH")
print(df_CH.GLACIER.unique())

# --- load aletsch monthly grid ---
glacier_name = 'aletsch'
rgi_id_aletsch = df_CH[df_CH.GLACIER == glacier_name].RGIId.values[0]

path_monthly_dfs = os.path.join(
    cfg.dataPath, 'RGI_v6/RGI_11_CentralEurope/monthly_parquets/')
path_parquet = os.path.join(path_monthly_dfs, rgi_id_aletsch)

yearly_files = sorted(
    glob.glob(os.path.join(path_parquet, f"{rgi_id_aletsch}_grid_*.parquet")))
print(f"Found {len(yearly_files)} yearly files for {glacier_name}")

df_aletsch = pd.concat([pd.read_parquet(f) for f in yearly_files],
                       ignore_index=True)
df_aletsch.dropna(subset=feature_columns, inplace=True)
print(f"Shape: {df_aletsch.shape}")

In [ ]:
df_aletsch.YEAR.unique()

In [ ]:
# monthly Swiss dataset:
output_file_monthly = os.path.join(
    cfg.dataPath, 'WGMS/Switzerland/csv/CH_wgms_dataset_all_monthly.csv')
df_CH['SOURCE_CODE'] = 'CH'
df_CH['RGI_REGION'] = '11'

data_monthly_CH = process_or_load_data(
    run_flag=False,
    df=df_CH,
    paths=paths,
    cfg=cfg,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=VOIS_TOPOGRAPHICAL,
    region_name="CH",
    region_id=11,
    add_pcsr=False,
    output_file=output_file_monthly,
)

In [ ]:
# ── Swiss bounding box ────────────────────────────────────────────────────────
CH_LON_MIN, CH_LON_MAX = 5.9, 10.5
CH_LAT_MIN, CH_LAT_MAX = 45.8, 47.8

CLIMATE_COLS = [
    't2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE'
]
TOPO_COLS = ['aspect', 'slope', 'svf']
ALL_COLS = CLIMATE_COLS + TOPO_COLS

# ── Load RGI shapefile and filter to Switzerland ──────────────────────────────
shapefile_path = os.path.join(
    cfg.dataPath, 'RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp')
gdf_rgi = gpd.read_file(shapefile_path)

gdf_CH = gdf_rgi[(gdf_rgi['CenLon'] >= CH_LON_MIN)
                 & (gdf_rgi['CenLon'] <= CH_LON_MAX) &
                 (gdf_rgi['CenLat'] >= CH_LAT_MIN) &
                 (gdf_rgi['CenLat'] <= CH_LAT_MAX)].copy()

swiss_rgi_ids = set(gdf_CH['RGIId'].values)
print(f"Swiss glaciers in RGI shapefile: {len(swiss_rgi_ids)}")

In [ ]:
# ── Filter by glacier size ────────────────────────────────────────────────────
MIN_AREA_KM2 = 0.5

gdf_CH_filtered = gdf_CH[gdf_CH['Area'] >= MIN_AREA_KM2].copy()
swiss_rgi_ids_filtered = set(gdf_CH_filtered['RGIId'].values)
print(
    f"Swiss glaciers after area filter (>= {MIN_AREA_KM2} km²): {len(swiss_rgi_ids_filtered)}"
)

# ── Load grids, last 5 years only ────────────────────────────────────────────
LAST_N_YEARS = 5

glacier_grids = {}
skipped_no_files = 0

for rgi_id in tqdm(swiss_rgi_ids_filtered, desc="Loading Swiss glacier grids"):
    yearly_files = sorted(
        glob.glob(
            os.path.join(path_monthly_dfs, rgi_id,
                         f"{rgi_id}_grid_*.parquet")))
    if not yearly_files:
        skipped_no_files += 1
        continue

    # Take last 5 yearly files
    yearly_files = yearly_files[-LAST_N_YEARS:]
    df_gl = pd.concat([pd.read_parquet(f) for f in yearly_files],
                      ignore_index=True)
    df_gl.dropna(subset=ALL_COLS, inplace=True)

    if len(df_gl) < 10:
        continue
    glacier_grids[rgi_id] = df_gl

print(
    f"\nSwiss glaciers in shapefile (>= {MIN_AREA_KM2} km²): {len(swiss_rgi_ids_filtered)}"
)
print(f"  — with grid files:    {len(glacier_grids)}")
print(f"  — no grid files:      {skipped_no_files}")
print(
    f"\nTotal rows across all glaciers: {sum(len(df) for df in glacier_grids.values()):,}"
)

In [ ]:
# ── 2. Prepare target: Swiss stake measurements ───────────────────────────────
df_target = data_monthly_CH.dropna(subset=ALL_COLS).copy()
print(f"Target (CH stakes): {len(df_target):,} rows")

In [ ]:
# Quick diagnostic before deciding on N_POINTS
for glacier_name, rgi_id in [('aletsch', rgi_id_aletsch)]:
    df = glacier_grids[rgi_id]
    n_points = df['POINT_ID'].nunique()
    n_years = df['YEAR'].nunique()
    n_rows = len(df)
    print(
        f"{glacier_name}: {n_points} unique points × {n_years} years × 12 months = {n_rows} rows"
    )

# And the distribution across all Swiss glaciers
point_counts = pd.Series({
    rgi_id: df['POINT_ID'].nunique()
    for rgi_id, df in glacier_grids.items()
})
print(f"\nUnique POINT_IDs per glacier:")
print(point_counts.describe().round(0))
print(f"\nSmallest glaciers: {point_counts.nsmallest(5).to_dict()}")
print(f"Largest glaciers:  {point_counts.nlargest(5).to_dict()}")

In [ ]:
# ── Load grids, last 5 years, proportional POINT_ID subsampling ───────────────
LAST_N_YEARS = 5
MIN_POINTS = 200
SAMPLE_FRAC = 0.2  # keep 20% of unique POINT_IDs, minimum 200

glacier_grids = {}
skipped_no_files = 0

for rgi_id in tqdm(swiss_rgi_ids_filtered, desc="Loading Swiss glacier grids"):
    yearly_files = sorted(
        glob.glob(
            os.path.join(path_monthly_dfs, rgi_id,
                         f"{rgi_id}_grid_*.parquet")))
    if not yearly_files:
        skipped_no_files += 1
        continue

    yearly_files = yearly_files[-LAST_N_YEARS:]
    df_gl = pd.concat([pd.read_parquet(f) for f in yearly_files],
                      ignore_index=True)
    df_gl.dropna(subset=ALL_COLS, inplace=True)

    if len(df_gl) < 10:
        continue

    # Proportional spatial subsampling by POINT_ID
    unique_points = df_gl['POINT_ID'].unique()
    n_sample = max(MIN_POINTS, int(len(unique_points) * SAMPLE_FRAC))
    n_sample = min(n_sample, len(unique_points))

    rng = np.random.default_rng(cfg.seed)
    sampled_points = rng.choice(unique_points, size=n_sample, replace=False)
    df_gl = df_gl[df_gl['POINT_ID'].isin(sampled_points)].copy()

    # ── Convert aspect & slope from radians to degrees ────────────────────────
    df_gl['aspect'] = np.degrees(df_gl['aspect'])
    df_gl['slope']  = np.degrees(df_gl['slope'])

    glacier_grids[rgi_id] = df_gl

print(f"Glaciers loaded: {len(glacier_grids)}")
print(f"Total rows after subsampling: {sum(len(df) for df in glacier_grids.values()):,}")

# Quick sanity check on Aletsch
df_al = glacier_grids[rgi_id_aletsch]
print(f"\nAletsch: {df_al['POINT_ID'].nunique()} points × "
      f"{df_al['YEAR'].nunique()} years = {len(df_al):,} rows")

# Sanity check units
print(f"\nGrid aspect range: {df_al['aspect'].min():.2f} – {df_al['aspect'].max():.2f}")
print(f"Grid slope range:  {df_al['slope'].min():.2f} – {df_al['slope'].max():.2f}")
print(f"Stake aspect range: {df_target['aspect'].min():.2f} – {df_target['aspect'].max():.2f}")
print(f"Stake slope range:  {df_target['slope'].min():.2f} – {df_target['slope'].max():.2f}")

In [ ]:
# ── Fit global scalers + bandwidths on full spatially-stratified data ─────────
all_dfs_full = {**glacier_grids, '__target__': df_target}
print(f"Total rows for scaler fitting: {sum(len(df) for df in all_dfs_full.values()):,}")

scaler_m, scaler_s, scaler_joint = build_global_scalers_from_dfs(
    dfs=all_dfs_full,
    monthly_cols=CLIMATE_COLS,
    static_cols=TOPO_COLS,
    topo_id_col='ID',
    glacier_col='GLACIER',
)

blur_m, blur_s, blur_joint = estimate_global_bandwidths_from_dfs(
    dfs=all_dfs_full,
    monthly_cols=CLIMATE_COLS,
    static_cols=TOPO_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)

bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
print(f"Global blurs — climate: {blur_m:.4f}, topo: {blur_s:.4f}, joint: {blur_joint:.4f}")

# ── Sanity check scaled distributions ────────────────────────────────────────
Xs_src_z = scaler_s.transform(
    glacier_grids[rgi_id_aletsch].drop_duplicates(subset=['GLACIER', 'ID'])[TOPO_COLS].to_numpy())
Xs_tgt_z = scaler_s.transform(
    df_target.drop_duplicates(subset=['GLACIER', 'ID'])[TOPO_COLS].to_numpy())
print(f"\nScaled topo src (Aletsch) — mean: {Xs_src_z.mean():.3f}, std: {Xs_src_z.std():.3f}")
print(f"Scaled topo tgt (stakes)  — mean: {Xs_tgt_z.mean():.3f}, std: {Xs_tgt_z.std():.3f}")

In [ ]:
import csv

out_path = os.path.join(cfg.dataPath, 'WGMS/Switzerland/csv/CH_glacier_sinkhorn_distances_2025_05_06.csv')

# ── Resume from existing file if present ─────────────────────────────────────
if os.path.exists(out_path):
    df_existing = pd.read_csv(out_path)
    completed_ids = set(df_existing['RGIId'].values)
    print(f"Resuming — {len(completed_ids)} glaciers already computed")
else:
    completed_ids = set()
    print("Starting fresh")

# ── Compute Sinkhorn distances incrementally ──────────────────────────────────
fieldnames = ['RGIId', 'D_sinkhorn_joint', 'D_sinkhorn_climate', 'D_sinkhorn_topo',
              'n_grid_points', 'n_rows']

with open(out_path, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    
    # Write header only if file is new
    if not completed_ids:
        writer.writeheader()

    for rgi_id, df_src in tqdm(glacier_grids.items(), desc="Computing Sinkhorn distances"):
        if rgi_id in completed_ids:
            continue

        shift = compute_domain_shift(
            df_src=df_src,
            df_tgt=df_target,
            monthly_cols=CLIMATE_COLS,
            static_cols=TOPO_COLS,
            scaler_m=scaler_m,
            scaler_s=scaler_s,
            scaler_joint=scaler_joint,
            blur_m=blur_m,
            blur_s=blur_s,
            blur_joint=blur_joint,
            bandwidths_m=bandwidths_m,
            bandwidths_s=bandwidths_s,
            seed=cfg.seed,
        )

        row = {
            'RGIId':              rgi_id,
            'D_sinkhorn_joint':   shift.get('D_sinkhorn_joint',   np.nan),
            'D_sinkhorn_climate': shift.get('D_sinkhorn_climate', np.nan),
            'D_sinkhorn_topo':    shift.get('D_sinkhorn_topo',    np.nan),
            'n_grid_points':      df_src['POINT_ID'].nunique(),
            'n_rows':             len(df_src),
        }
        writer.writerow(row)
        f.flush()  # force write to disk after each glacier

# ── Load and summarise final results ─────────────────────────────────────────
df_distances = pd.read_csv(out_path).sort_values('D_sinkhorn_joint')
print(f"\nCompleted: {len(df_distances)} / {len(glacier_grids)} glaciers")
print("\nTop 10 closest to CH stake distribution (joint):")
print(df_distances.head(10).to_string(index=False))
print("\nTop 10 most distant (joint):")
print(df_distances.tail(10).to_string(index=False))

In [ ]:
# ── Join distances with shapefile ─────────────────────────────────────────────
df_distances = pd.read_csv(out_path)
gdf_plot = gdf_CH_filtered.merge(df_distances, on='RGIId', how='inner')

# ── Exclude glaciers that already have stake measurements ─────────────────────
rgi_ids_with_stakes = set(data_monthly_CH['RGIId'].unique())
gdf_plot_novel = gdf_plot[~gdf_plot['RGIId'].isin(rgi_ids_with_stakes)].copy()
gdf_stakes = gdf_plot[gdf_plot['RGIId'].isin(rgi_ids_with_stakes)].copy()
print(f"Glaciers with stakes:    {len(gdf_stakes)}")
print(f"Glaciers without stakes: {len(gdf_plot_novel)}")

# ── Plot settings (Nature style) ──────────────────────────────────────────────
NATURE_SINGLE_COL = 3.5
plt.rcParams.update({
    'font.size': 7, 'axes.labelsize': 7, 'xtick.labelsize': 6,
    'ytick.labelsize': 6, 'legend.fontsize': 6, 'axes.linewidth': 0.5,
})

metrics = [
    ('D_sinkhorn_joint',   'Joint'),
    ('D_sinkhorn_climate', 'Climate'),
    ('D_sinkhorn_topo',    'Topographic'),
]

cmap = 'flare'
LON_BUF, LAT_BUF = 0.3, 0.2
extent = [6.5, CH_LON_MAX + LON_BUF, CH_LAT_MIN - LAT_BUF, 47.1]
proj = ccrs.PlateCarree()

area_min, area_max = gdf_plot['Area'].min(), gdf_plot['Area'].max()
size_min, size_max = 100, 400
def scale_area(area):
    return size_min + (area - area_min) / (area_max - area_min) * (size_max - size_min)

size_legend_areas = [1, 10, 50]
os.makedirs('figures/paperTF/CH_sinkhorn_maps', exist_ok=True)

# store norms per metric for reuse in zoom cell
norms = {}

for metric, label in metrics:
    fig, ax = plt.subplots(1, 1,
        figsize=(NATURE_SINGLE_COL * 2.5, NATURE_SINGLE_COL * 1.5),
        subplot_kw={'projection': proj},
    )
    ax.set_extent(extent, crs=proj)
    ax.set_aspect('auto')
    ax.add_feature(cfeature.LAND,    facecolor='#f5f5f5', zorder=0)
    ax.add_feature(cfeature.OCEAN,   facecolor='#d6eaf8', zorder=0)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor='#aaaaaa', zorder=1)
    ax.add_feature(cfeature.RIVERS,  linewidth=0.3, edgecolor='#aec6cf', zorder=1)
    ax.add_feature(cfeature.LAKES,   facecolor='#d6eaf8', zorder=1)
    ax.coastlines(resolution='10m',  linewidth=0.4, zorder=2)
    ax.add_geometries(
        gdf_CH_filtered['geometry'], crs=proj,
        facecolor='grey', edgecolor='darkgrey', linewidth=0.2, zorder=2, alpha=0.6,
    )

    vmin = gdf_plot_novel[metric].quantile(0.05)
    vmax = gdf_plot_novel[metric].quantile(0.95)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    norms[metric] = norm

    sc = ax.scatter(
        gdf_plot_novel['CenLon'], gdf_plot_novel['CenLat'],
        c=gdf_plot_novel[metric], s=scale_area(gdf_plot_novel['Area']),
        cmap=cmap, norm=norm, transform=proj,
        edgecolors='white', linewidths=0.2, zorder=3, alpha=0.85,
    )
    ax.scatter(
        gdf_stakes['CenLon'], gdf_stakes['CenLat'],
        s=200, marker='*', color='black', transform=proj,
        edgecolors='white', linewidths=0.2, zorder=4, alpha=0.9,
    )

    ax.set_title(label, fontsize=7, fontweight='bold', pad=4)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                      alpha=0.5, linestyle='--')
    gl.top_labels   = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 5}
    gl.ylabel_style = {'size': 5}

    plt.tight_layout()
    fig_path = f'figures/paperTF/CH_sinkhorn_maps/CH_sinkhorn_{metric}.pdf'
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    print(f"Saved {fig_path}")
    plt.show()
    plt.close(fig)

    # ── Colorbar ──────────────────────────────────────────────────────────────
    fig_cbar, ax_cbar = plt.subplots(figsize=(2.5, 0.25))
    fig_cbar.colorbar(sc, cax=ax_cbar, orientation='horizontal')
    ax_cbar.tick_params(labelsize=5)
    ax_cbar.set_xlabel(f'Sinkhorn distance ({label.lower()})', fontsize=6)
    fig_cbar.savefig(f'figures/paperTF/CH_sinkhorn_maps/CH_sinkhorn_{metric}_colorbar.pdf',
                     dpi=300, bbox_inches='tight')
    plt.close(fig_cbar)

    # ── Size legend ───────────────────────────────────────────────────────────
    size_legend_handles = [
        Line2D([0], [0], marker='o', color='w',
               markerfacecolor='grey', markersize=np.sqrt(scale_area(a)),
               label=f'{a} km²')
        for a in size_legend_areas
    ] + [
        Line2D([0], [0], marker='*', color='w',
               markerfacecolor='black', markersize=6,
               label='stake measurements')
    ]
    fig_leg, ax_leg = plt.subplots(figsize=(2.5, 0.4))
    ax_leg.axis('off')
    ax_leg.legend(handles=size_legend_handles, title='Glacier area', loc='center',
                  frameon=False, fontsize=5, title_fontsize=5,
                  ncols=len(size_legend_areas) + 1)
    fig_leg.savefig(f'figures/paperTF/CH_sinkhorn_maps/CH_sinkhorn_{metric}_legend.pdf',
                    dpi=300, bbox_inches='tight')
    plt.close(fig_leg)
    print(f"Saved legend and colorbar for {metric}")

### PPT version:

In [ ]:
# ── Join distances with shapefile ─────────────────────────────────────────────
df_distances = pd.read_csv(out_path)
gdf_plot = gdf_CH_filtered.merge(df_distances, on='RGIId', how='inner')

# ── Exclude glaciers that already have stake measurements ─────────────────────
rgi_ids_with_stakes = set(data_monthly_CH['RGIId'].unique())
gdf_plot_novel = gdf_plot[~gdf_plot['RGIId'].isin(rgi_ids_with_stakes)].copy()
print(
    f"Glaciers with stakes:    {gdf_plot['RGIId'].isin(rgi_ids_with_stakes).sum()}"
)
print(f"Glaciers without stakes: {len(gdf_plot_novel)}")

# ── Plot settings (Nature style) ──────────────────────────────────────────────
NATURE_SINGLE_COL = 3.5
plt.rcParams.update({
    'font.size': 7,
    'axes.labelsize': 7,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 6,
    'axes.linewidth': 0.5,
})

metrics = [
    ('D_sinkhorn_joint', 'Joint'),
    ('D_sinkhorn_climate', 'Climate'),
    ('D_sinkhorn_topo', 'Topographic'),
]

cmap = 'flare'
LON_BUF, LAT_BUF = 0.3, 0.2
extent = [6.5, CH_LON_MAX + LON_BUF, CH_LAT_MIN - LAT_BUF, 47.1]
proj = ccrs.PlateCarree()

area_min, area_max = gdf_plot['Area'].min(), gdf_plot['Area'].max()
size_min, size_max = 100, 400


def scale_area(area):
    return size_min + (area - area_min) / (area_max - area_min) * (size_max -
                                                                   size_min)


size_legend_areas = [1, 10, 50]
gdf_stakes = gdf_plot[gdf_plot['RGIId'].isin(rgi_ids_with_stakes)].copy()
os.makedirs('figures/paperTF/CH_sinkhorn_maps', exist_ok=True)

# store norms per metric so zoomed cell can reuse them
norms = {}

for metric, label in metrics:
    fig, ax = plt.subplots(
        1,
        1,
        figsize=(NATURE_SINGLE_COL * 2.5, NATURE_SINGLE_COL * 1.5),
        subplot_kw={'projection': proj},
    )
    ax.set_extent(extent, crs=proj)
    ax.set_aspect('auto')
    ax.add_feature(cfeature.LAND, facecolor='#f5f5f5', zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor='#d6eaf8', zorder=0)
    ax.add_feature(cfeature.BORDERS,
                   linewidth=0.4,
                   edgecolor='#aaaaaa',
                   zorder=1)
    ax.add_feature(cfeature.RIVERS,
                   linewidth=0.3,
                   edgecolor='#aec6cf',
                   zorder=1)
    ax.add_feature(cfeature.LAKES, facecolor='#d6eaf8', zorder=1)
    ax.coastlines(resolution='10m', linewidth=0.4, zorder=2)
    ax.add_geometries(
        gdf_CH_filtered['geometry'],
        crs=proj,
        facecolor='grey',
        edgecolor='darkgrey',
        linewidth=0.2,
        zorder=2,
        alpha=0.6,
    )

    vmin = gdf_plot_novel[metric].quantile(0.05)
    vmax = gdf_plot_novel[metric].quantile(0.95)
    print(metric, vmin, vmax)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    norms[metric] = norm  # store for zoomed cell

    sc = ax.scatter(
        gdf_plot_novel['CenLon'],
        gdf_plot_novel['CenLat'],
        c=gdf_plot_novel[metric],
        s=scale_area(gdf_plot_novel['Area']),
        cmap=cmap,
        norm=norm,
        transform=proj,
        edgecolors='white',
        linewidths=0.2,
        zorder=3,
        alpha=0.85,
    )
    ax.scatter(
        gdf_stakes['CenLon'],
        gdf_stakes['CenLat'],
        s=200,
        marker='*',
        color='black',
        transform=proj,
        edgecolors='white',
        linewidths=0.2,
        zorder=4,
        alpha=0.9,
    )

    ax.set_title(label, fontsize=7, fontweight='bold', pad=4)
    gl = ax.gridlines(draw_labels=True,
                      linewidth=0.3,
                      color='gray',
                      alpha=0.5,
                      linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 5}
    gl.ylabel_style = {'size': 5}

    plt.tight_layout()
    fig_path = f'figures/paperTF/CH_sinkhorn_maps/CH_sinkhorn_{metric}.pdf'
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    print(f"Saved {fig_path}")
    plt.show()
    plt.close(fig)

    # ── Colorbar ──────────────────────────────────────────────────────────────
    fig_cbar, ax_cbar = plt.subplots(figsize=(2.5, 0.25))
    fig_cbar.colorbar(sc, cax=ax_cbar, orientation='horizontal')
    ax_cbar.tick_params(labelsize=5)
    ax_cbar.set_xlabel(f'Sinkhorn distance ({label.lower()})', fontsize=6)
    fig_cbar.savefig(
        f'figures/paperTF/CH_sinkhorn_maps/CH_sinkhorn_{metric}_colorbar.pdf',
        dpi=300,
        bbox_inches='tight')
    plt.close(fig_cbar)
    print(
        f"Saved figures/paperTF/CH_sinkhorn_maps/CH_sinkhorn_{metric}_colorbar.pdf"
    )

    # ── Size legend ───────────────────────────────────────────────────────────
    size_legend_handles = [
        Line2D([0], [0],
               marker='o',
               color='w',
               markerfacecolor='grey',
               markersize=np.sqrt(scale_area(a)),
               label=f'{a} km²') for a in size_legend_areas
    ]
    fig_leg, ax_leg = plt.subplots(figsize=(2.5, 0.4))
    ax_leg.axis('off')
    ax_leg.legend(handles=size_legend_handles,
                  title='Glacier area',
                  loc='center',
                  frameon=False,
                  fontsize=5,
                  title_fontsize=5,
                  ncols=len(size_legend_areas))
    fig_leg.savefig(
        f'figures/paperTF/CH_sinkhorn_maps/CH_sinkhorn_{metric}_legend.pdf',
        dpi=300,
        bbox_inches='tight')
    plt.close(fig_leg)
    print(
        f"Saved figures/paperTF/CH_sinkhorn_maps/CH_sinkhorn_{metric}_legend.pdf"
    )

In [ ]:
# ── Zoomed-in version around Aletsch ─────────────────────────────────────────
ALETSCH_EXTENT = [7.6, 8.4, 46.3, 46.7]

for metric, label in metrics:
    fig_zoom, ax_zoom = plt.subplots(
        1,
        1,
        figsize=(NATURE_SINGLE_COL * 2.5, NATURE_SINGLE_COL * 1.5),
        subplot_kw={'projection': proj},
    )
    ax_zoom.set_extent(ALETSCH_EXTENT, crs=proj)
    ax_zoom.set_aspect('auto')
    ax_zoom.add_feature(cfeature.LAND, facecolor='#f5f5f5', zorder=0)
    ax_zoom.add_feature(cfeature.OCEAN, facecolor='#d6eaf8', zorder=0)
    ax_zoom.add_feature(cfeature.BORDERS,
                        linewidth=0.4,
                        edgecolor='#aaaaaa',
                        zorder=1)
    ax_zoom.add_feature(cfeature.RIVERS,
                        linewidth=0.3,
                        edgecolor='#aec6cf',
                        zorder=1)
    ax_zoom.add_feature(cfeature.LAKES, facecolor='#d6eaf8', zorder=1)
    ax_zoom.coastlines(resolution='10m', linewidth=0.4, zorder=2)

    # ── Color novel glacier polygons by distance ───────────────────────────────
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norms[metric])
    for _, row in gdf_plot_novel.iterrows():
        color = sm.to_rgba(row[metric])
        ax_zoom.add_geometries(
            [row['geometry']],
            crs=proj,
            facecolor=color,
            edgecolor='white',
            linewidth=0.2,
            zorder=3,
            alpha=0.85,
        )

    # ── Stake glaciers in grey (no distance computed for them) ────────────────
    ax_zoom.add_geometries(
        gdf_stakes['geometry'],
        crs=proj,
        facecolor='grey',
        edgecolor='darkgrey',
        linewidth=0.2,
        zorder=3,
        alpha=0.6,
    )

    # ── Stars on stake glacier centroids ──────────────────────────────────────
    ax_zoom.scatter(
        gdf_stakes['CenLon'],
        gdf_stakes['CenLat'],
        s=200,
        marker='*',
        color='black',
        transform=proj,
        edgecolors='white',
        linewidths=0.2,
        zorder=4,
        alpha=0.9,
    )

    ax_zoom.set_title(f'{label} — Aletsch region',
                      fontsize=7,
                      fontweight='bold',
                      pad=4)
    gl_zoom = ax_zoom.gridlines(draw_labels=True,
                                linewidth=0.3,
                                color='gray',
                                alpha=0.5,
                                linestyle='--')
    gl_zoom.top_labels = False
    gl_zoom.right_labels = False
    gl_zoom.xlabel_style = {'size': 5}
    gl_zoom.ylabel_style = {'size': 5}

    plt.tight_layout()
    fig_zoom_path = f'figures/paperTF/CH_sinkhorn_maps/CH_sinkhorn_{metric}_zoom_aletsch.pdf'
    plt.savefig(fig_zoom_path, dpi=300, bbox_inches='tight')
    print(f"Saved {fig_zoom_path}")
    plt.show()
    plt.close(fig_zoom)

In [ ]:
# ── Find glaciers within Aletsch extent ───────────────────────────────────────
ALETSCH_EXTENT = [7.6, 8.4, 46.3, 46.7]
lon_min, lon_max, lat_min, lat_max = ALETSCH_EXTENT

gdf_aletsch_region = gdf_plot_novel[
    (gdf_plot_novel['CenLon'] >= lon_min)
    & (gdf_plot_novel['CenLon'] <= lon_max) &
    (gdf_plot_novel['CenLat'] >= lat_min) &
    (gdf_plot_novel['CenLat'] <= lat_max)].sort_values('D_sinkhorn_topo')

print(f"Novel glaciers in Aletsch region: {len(gdf_aletsch_region)}")
print(gdf_aletsch_region[[
    'RGIId', 'Name', 'Area', 'D_sinkhorn_joint', 'D_sinkhorn_climate',
    'D_sinkhorn_topo'
]].to_string(index=False))

In [ ]:
def format_axis_ticks(ax, label_size=8):
    """Format tick labels to avoid huge 1e6/1e7 offset labels."""
    # check if scientific notation offset is being used
    ax.xaxis.get_major_formatter().set_useOffset(False)
    try:
        # scale large numbers to readable units
        xmax = abs(ax.get_xlim()[1])
        if xmax > 1e6:
            scale = 1e6
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{x/scale:.1f}"))
            ax.set_xlabel(f"(×10⁶)", label_size=label_size, labelpad=1)
        elif xmax > 1e4:
            scale = 1e3
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{x/scale:.0f}"))
            ax.set_xlabel(f"(×10³)", label_size=label_size, labelpad=1)
    except Exception:
        pass


def plot_kde_pair(glaciers_to_plot, selected_cols, save_prefix):
    """KDE panels for a pair of glaciers."""
    ncols = 3
    nrows = int(np.ceil(len(selected_cols) / ncols))
    w, h = nature_figsize(cols=1, height_mm=80)
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * 2, h), squeeze=False)

    for idx, col in enumerate(selected_cols):
        ax = axes[idx // ncols][idx % ncols]

        all_vals = pd.concat([
            cfg_gl["df"][col].dropna() for cfg_gl in glaciers_to_plot.values()
        ])
        x_grid = np.linspace(float(all_vals.min()), float(all_vals.max()), 500)

        for label, cfg_gl in glaciers_to_plot.items():
            vals = cfg_gl["df"][col].dropna().values
            if len(vals) < 10:
                continue
            kde = scipy_stats.gaussian_kde(vals, bw_method=0.3)
            y = kde(x_grid)
            y = y / y.max()
            ax.plot(x_grid,
                    y,
                    color=cfg_gl["color"],
                    linewidth=0.8,
                    label=label)
            ax.fill_between(x_grid, y, alpha=0.08, color=cfg_gl["color"])

        ax.set_ylim(0, 1.15)
        ax.set_xlabel("")
        ax.set_title(col, fontsize=6)
        ax.tick_params(labelsize=8, width=0.4, length=2, direction="in")
        ax.spines[["top", "right", "left", "bottom"]].set_visible(True)
        for spine in ax.spines.values():
            spine.set_linewidth(0.4)
        ax.grid(axis="x", color="#e0e0e0", linewidth=0.3)
        ax.set_axisbelow(True)
        format_axis_ticks(ax, label_size=6)

    for idx in range(len(selected_cols), nrows * ncols):
        axes[idx // ncols][idx % ncols].axis("off")

    # ── Legend ────────────────────────────────────────────────────────────────
    handles = [
        plt.Line2D([0], [0], color=cfg_gl["color"], linewidth=1.5, label=label)
        for label, cfg_gl in glaciers_to_plot.items()
    ]
    fig.legend(
        handles=handles,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.04),
        ncols=len(glaciers_to_plot),
        frameon=False,
        fontsize=6,
    )

    plt.tight_layout(h_pad=3.0)
    plt.savefig(f"figures/paperTF/{save_prefix}_kde.pdf", bbox_inches="tight")
    plt.savefig(f"figures/paperTF/{save_prefix}_kde.png",
                dpi=300,
                bbox_inches="tight")
    plt.show()


def plot_tsne_pair(glaciers_to_plot,
                   monthly_cols,
                   static_cols,
                   save_prefix,
                   n_samples=500,
                   max_iter=1000,
                   perplexity=30):
    """Three-panel t-SNE (joint, climate-only, topo-only) for a pair of glaciers."""

    panel_defs = [
        ("Joint\n(climate + topo)", monthly_cols + static_cols),
        ("Climate only", monthly_cols),
        ("Topographic only", static_cols),
    ]

    all_cols = monthly_cols + static_cols
    frames, labels_list = [], []
    for label, cfg_gl in glaciers_to_plot.items():
        df = cfg_gl["df"].dropna(subset=all_cols)
        df_s = df.sample(min(n_samples, len(df)), random_state=cfg.seed)
        frames.append(df_s)
        labels_list.extend([label] * len(df_s))

    df_all = pd.concat(frames, ignore_index=True)
    labels_arr = np.array(labels_list)

    w, h = nature_figsize(cols=2, height_mm=55)
    fig, axes = plt.subplots(1, 3, figsize=(w, h))

    for ax, (title, feat_cols) in zip(axes, panel_defs):
        X = StandardScaler().fit_transform(df_all[feat_cols].values)
        perp = min(perplexity, len(X) // 4)
        emb = TSNE(n_components=2,
                   perplexity=perp,
                   max_iter=max_iter,
                   random_state=cfg.seed).fit_transform(X)

        for label, cfg_gl in glaciers_to_plot.items():
            mask = labels_arr == label
            ax.scatter(emb[mask, 0],
                       emb[mask, 1],
                       c=cfg_gl["color"],
                       label=label,
                       alpha=0.4,
                       s=6,
                       linewidths=0,
                       rasterized=True)

        ax.set_title(title, fontsize=NATURE_SPECS["font_max_pt"])
        ax.set_xlabel("t-SNE 1", fontsize=NATURE_SPECS["font_max_pt"])
        ax.set_ylabel("t-SNE 2", fontsize=NATURE_SPECS["font_max_pt"])
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)

    handles = [
        plt.Line2D([0], [0],
                   marker="o",
                   color="w",
                   markerfacecolor=cfg_gl["color"],
                   markersize=4,
                   label=label) for label, cfg_gl in glaciers_to_plot.items()
    ]
    # fig.legend(handles=handles,
    #            loc="lower center",
    #            ncol=len(glaciers_to_plot),
    #            fontsize=NATURE_SPECS["font_min_pt"],
    #            frameon=False,
    #            bbox_to_anchor=(0.5, -0.08))

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.savefig(f"figures/paperTF/{save_prefix}_tsne.pdf", bbox_inches="tight")
    plt.savefig(f"figures/paperTF/{save_prefix}_tsne.png",
                dpi=NATURE_SPECS["dpi"],
                bbox_inches="tight")
    plt.show()

In [ ]:
# ── Load grid for RGI60-11.01885 ──────────────────────────────────────────────
rgi_id_compare = 'RGI60-11.01885'
df_compare = glacier_grids[rgi_id_compare]
glacier_name_compare = df_compare['GLACIER'].iloc[0]
print(f"Glacier: {glacier_name_compare}, n_rows: {len(df_compare):,}")

# ── Build glaciers_to_plot dict ───────────────────────────────────────────────
glaciers_to_plot = {
    f'{glacier_name_compare} (grid)': {
        'df': df_compare,
        'color': NATURE_PALETTE['blue'],
    },
    'CH stakes': {
        'df': df_target,
        'color': NATURE_PALETTE['vermillion'],
    },
}

# ── Plot KDEs ─────────────────────────────────────────────────────────────────
feature_columns = CLIMATE_COLS + TOPO_COLS

SELECTED_COLS = [
    't2m',
    'tp',
    'ssrd',
    'ELEVATION_DIFFERENCE',
    'aspect',
    'slope',
]

plot_kde_pair(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=SELECTED_COLS,
    save_prefix=f'CH_vs_{rgi_id_compare}',
)

In [ ]:
# ── Load grid for RGI60-11.01885 ──────────────────────────────────────────────
rgi_id_compare = 'RGI60-11.01702'
df_compare = glacier_grids[rgi_id_compare]
glacier_name_compare = df_compare['GLACIER'].iloc[0]
print(f"Glacier: {glacier_name_compare}, n_rows: {len(df_compare):,}")

# ── Build glaciers_to_plot dict ───────────────────────────────────────────────
glaciers_to_plot = {
    f'{glacier_name_compare} (grid)': {
        'df': df_compare,
        'color': NATURE_PALETTE['blue'],
    },
    'CH stakes': {
        'df': df_target,
        'color': NATURE_PALETTE['vermillion'],
    },
}

# ── Plot KDEs ─────────────────────────────────────────────────────────────────
feature_columns = CLIMATE_COLS + TOPO_COLS

SELECTED_COLS = [
    't2m',
    'tp',
    'ssrd',
    'ELEVATION_DIFFERENCE',
    'aspect',
    'slope',
]

plot_kde_pair(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=SELECTED_COLS,
    save_prefix=f'CH_vs_{rgi_id_compare}',
)

In [ ]:
glaciers_to_plot_aletsch = {
    'Aletsch (grid)': {
        'df': glacier_grids[rgi_id_aletsch],
        'color': NATURE_PALETTE['blue'],
    },
    'CH stakes': {
        'df': df_target,
        'color': NATURE_PALETTE['vermillion'],
    },
}

plot_kde_pair(
    glaciers_to_plot=glaciers_to_plot_aletsch,
    selected_cols=SELECTED_COLS,
    save_prefix='CH_vs_aletsch',
)

In [ ]:
rgi_id_gl = df_CH[df_CH.GLACIER == 'gietro'].RGIId.values[0]
glaciers_to_plot_gietro = {
    'Gietro (grid)': {
        'df': glacier_grids[rgi_id_gl],
        'color': NATURE_PALETTE['blue'],
    },
    'CH stakes': {
        'df': df_target,
        'color': NATURE_PALETTE['vermillion'],
    },
}

plot_kde_pair(
    glaciers_to_plot=glaciers_to_plot_gietro,
    selected_cols=SELECTED_COLS,
    save_prefix='CH_vs_aletsch',
)